# Colab setup with persistent Google Drive results

This notebook is a thin wrapper around the repository's existing `src` and `scripts`.

**Persistence/resume behavior:**
- All experiment outputs are written **directly to Google Drive**, not `/content`.
- Each completed run is therefore persisted immediately.
- The repository scripts are run with `SKIP_EXISTING=1`, so after a Colab disconnect you can reconnect, rerun the setup/upload cells, and rerun the experiment cell. Already-completed runs are skipped and execution continues from the remaining runs.
- The three datasets themselves are still uploaded to the Colab session each time, as requested.

The notebooks cover the three paper datasets: TelecomTS, MIT Supercloud GPU utilization, and ESNet `r-nova-fd-01`.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, subprocess, sys, shutil, pandas as pd, json

# Mount Google Drive first so results are persistent from the first completed run onward.
drive.mount('/content/drive')

# Change this only if you want a different Drive folder.
DRIVE_ROOT = Path('/content/drive/MyDrive/SystemsExcursionForecastability')
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

REPO_URL = 'PASTE_GIT_REPOSITORY_URL_HERE'
REPO_DIR = Path('/content/SystemsExcursionForecastability')

if not REPO_DIR.exists():
    if REPO_URL == 'PASTE_GIT_REPOSITORY_URL_HERE':
        raise ValueError('Set REPO_URL before running this cell.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

print('Repository ready:', REPO_DIR)
print('Persistent results root:', DRIVE_RESULTS)

In [ ]:
# Upload ALL THREE prepared CSVs to this Colab runtime.
# They do not need to be in Drive; only results are persisted there.
uploaded = files.upload()

DATA_DIR = REPO_DIR / 'data' / 'processed'
DATA_DIR.mkdir(parents=True, exist_ok=True)

def find_uploaded(*needles):
    matches = [n for n in uploaded if all(x.lower() in n.lower() for x in needles)]
    if len(matches) != 1:
        raise ValueError(
            f'Expected exactly one uploaded file matching {needles}; found {matches}; '
            f'uploaded={list(uploaded)}'
        )
    return matches[0]

telecom_upload = find_uploaded('telecom')
mit_upload = find_uploaded('mit')
esnet_upload = find_uploaded('esnet')

TELECOM_CSV = DATA_DIR / 'telecomts.csv'
MIT_CSV = DATA_DIR / 'mit_supercloud_gpu_utilization_gpu_pct.csv'
ESNET_CSV = DATA_DIR / 'esnet_r-nova-fd-01_delta.csv'

for src, dst in [
    (telecom_upload, TELECOM_CSV),
    (mit_upload, MIT_CSV),
    (esnet_upload, ESNET_CSV),
]:
    shutil.copy2(src, dst)

print('Uploaded datasets:')
print('  TelecomTS:', TELECOM_CSV)
print('  MIT:', MIT_CSV)
print('  ESNet:', ESNET_CSV)

In [ ]:
TELECOM_SERIES = 'telecomts::normal/stationary/Zone_B/no_congestion/File::TX_Bytes'
MIT_SERIES = 'mit_gpu::67691045781943-r7343737-n685852::utilization_gpu_pct'
ESNET_SERIES = 'esnet::r-nova-fd-01::ifHCOutOctets'

TELECOM_SIGNALS = ','.join([
    'TX_Bytes', 'RX_Bytes', 'Estimated_UL_Buffer',
    'PRB_Utilization_DL', 'PRB_Utilization_UL'
])
ESNET_SIGNALS = ','.join([
    'ifHCOutOctets', 'ifHCOutUcastPkts', 'ifHCOutMulticastPkts',
    'ifHCOutBroadcastPkts', 'ifHCInOctets', 'ifHCInUcastPkts',
    'ifHCInMulticastPkts', 'ifHCInBroadcastPkts'
])

# Multi-trace breadth experiment

For robustness to disconnects, each dataset/mode combination is handled as a separate job rather than combining all manifests into one long invocation.

For each job the notebook:
1. generates the canonical manifest with `ptaf trace-manifest`;
2. stores that manifest in Google Drive immediately;
3. invokes `scripts/run_multitrace_sweep.py` using a Drive-backed `OUTPUT_ROOT`;
4. writes a completion marker when that dataset/mode job finishes.

Individual forecasting run directories are also written directly to Drive, so an interruption partway through a dataset does not discard completed runs.

In [ ]:
MANIFEST_DIR = DRIVE_RESULTS / 'breadth_manifests'
BREADTH_OUTPUT = DRIVE_RESULTS / 'breadth'
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
BREADTH_OUTPUT.mkdir(parents=True, exist_ok=True)

manifest_jobs = [
    ('telecom_uni', TELECOM_CSV, 'TX_Bytes', 'univariate', None),
    ('telecom_multi', TELECOM_CSV, 'TX_Bytes', 'multivariate', TELECOM_SIGNALS.split(',')),
    ('mit_uni', MIT_CSV, 'utilization_gpu_pct', 'univariate', None),
    ('esnet_uni', ESNET_CSV, 'ifHCOutOctets', 'univariate', None),
    ('esnet_multi', ESNET_CSV, 'ifHCOutOctets', 'multivariate', ESNET_SIGNALS.split(',')),
]

os.environ['OUTPUT_ROOT'] = str(BREADTH_OUTPUT)
os.environ['SKIP_EXISTING'] = '1'

In [ ]:
for name, data_path, target_signal, mode, signals in manifest_jobs:
    print(f'\n===== {name} =====', flush=True)
    manifest = MANIFEST_DIR / f'{name}.csv'

    # Re-generate deterministically from the current source. This is cheap and ensures
    # the manifest matches the uploaded dataset and current code.
    manifest_cmd = [
        'ptaf', 'trace-manifest',
        '--data', str(data_path),
        '--target-signal', target_signal,
        '--n-traces', '3',
        '--mode', mode,
        '--input-size', '336',
        '--max-horizon', '720',
        '--output', str(manifest),
    ]
    if signals:
        manifest_cmd += ['--signals', *signals]

    print('>>>', ' '.join(manifest_cmd), flush=True)
    subprocess.run(manifest_cmd, check=True)
    print('Manifest persisted:', manifest)

    sweep_cmd = [
        sys.executable,
        str(REPO_DIR / 'scripts' / 'run_multitrace_sweep.py'),
        str(manifest),
        '--max-steps', '1000',
        '--tuning-max-steps', '100',
        '--horizons', '96 192 336 720',
        '--seeds', '1 2 3',
        '--univariate-models', 'dlinear timemixer',
        '--multivariate-models', 'itransformer timemixer',
        '--losses', 'mse mep',
    ]

    print('>>>', ' '.join(sweep_cmd), flush=True)
    subprocess.run(sweep_cmd, check=True, env=os.environ.copy())

    marker = BREADTH_OUTPUT / f'_COMPLETED_{name}.txt'
    marker.write_text('completed\n')
    print('Persisted completion marker:', marker)

## Resume behavior

If Colab disconnects, rerun from the top and re-upload the three CSVs. The same deterministic manifests are recreated, and `SKIP_EXISTING=1` causes the sweep script to skip already completed individual runs in the Drive output directory.